# 09 - Word-level error correction

Started as a catastrophic-line scan of the current best pipeline
(`crnn_residual_aug_elastic.pt` + CTC beam search + 4-gram char LM,
alpha=0.35, score 0.826): 17/120 val lines with `dist > 10` account for
47% of total edit distance - near-total mistranscriptions, not fixable by
character-level rules.

Reviewing those lines by hand showed the model often produces a fluent-
looking but non-existent *word* (`suney`, `tene`, `Portesgne` instead of
`surveyor`, `tenne`, `Fortesque`) rather than scattered letter noise - a
word-level lexicon check, not a character-level one, is the right tool.
The rest of this notebook: measure how many predicted words are out-of-
vocabulary against the fit split, build an objective (not hand-picked)
"confident correction" rule, and gate-check the result against the
current best score.

In [ ]:
import sys
sys.path.append('..')

import math
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from PIL import Image
from rapidfuzz.distance import Levenshtein
from torch.utils.data import DataLoader, Dataset

from src.config import RANDOM_STATE, MODELS_DIR
from src.data import build_image_path, load_raw_data, split_train_val
from src.evaluate import combined_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Reload data, vocab, and val sample (same fixed sample as notebooks 05/07/08)

In [ ]:
train = load_raw_data('train')
fit_df, val_df = split_train_val(train)

# Same fixed sample/seed as notebooks 05, 07, 08 - keeps this scan comparable.
val_sample = val_df.sample(120, random_state=RANDOM_STATE).reset_index(drop=True)
len(fit_df), len(val_df), len(val_sample)

In [ ]:
# Character vocabulary from the fit split - deterministic given fit_df,
# matches what crnn_residual_aug_elastic.pt was trained with.
chars = sorted(set(''.join(fit_df['Target'].tolist())))
BLANK = 0
char_to_idx = {c: i + 1 for i, c in enumerate(chars)}
idx_to_char = {i + 1: c for i, c in enumerate(chars)}
NUM_CLASSES = len(chars) + 1
NUM_CLASSES

In [ ]:
TARGET_HEIGHT = 32
MAX_WIDTH = 1024


def load_and_resize(record_id):
    img = Image.open(build_image_path(record_id)).convert('L')
    w, h = img.size
    new_w = max(8, min(MAX_WIDTH, round(w * TARGET_HEIGHT / h)))
    img = img.resize((new_w, TARGET_HEIGHT), Image.BILINEAR)
    arr = (np.asarray(img, dtype=np.float32) / 255.0 - 0.5) / 0.5
    return torch.from_numpy(arr).unsqueeze(0)  # 1 x H x W


def naive_output_len(w):
    """CTC input-length formula matching CRNNResidual's two plain
    MaxPool2d(2, 2) stages (unchanged from the baseline - residual blocks
    don't touch downsampling, see notebooks/07's per-variant note)."""
    return (w // 2) // 2


class LineDataset(Dataset):
    def __init__(self, df):
        self.ids = df['ID'].tolist()
        self.targets = df['Target'].tolist()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        img = load_and_resize(self.ids[i])
        text = self.targets[i]
        label = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)
        return img, label, text


def collate_fn(batch):
    imgs, labels, texts = zip(*batch)
    widths = [im.shape[-1] for im in imgs]
    max_w = max(widths)
    padded = torch.zeros(len(imgs), 1, TARGET_HEIGHT, max_w)
    for i, im in enumerate(imgs):
        padded[i, :, :, : im.shape[-1]] = im
    input_lengths = torch.tensor([naive_output_len(w) for w in widths], dtype=torch.long)
    target_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    return padded, torch.cat(labels), input_lengths, target_lengths, texts


val_dl = DataLoader(LineDataset(val_sample), batch_size=16, shuffle=False, collate_fn=collate_fn)

## Load the winning architecture and checkpoint (identical to notebooks 07/08)

In [ ]:
class ResidualBlock(nn.Module):
    """Basic residual block (He et al. 2015 - see RESOURCES.md)."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = (
            nn.Sequential(nn.Conv2d(in_channels, out_channels, 1), nn.BatchNorm2d(out_channels))
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class CRNNResidual(nn.Module):
    """Winning architecture: each conv block wrapped in a ResidualBlock,
    MaxPool2d downsampling unchanged. Identical to notebooks/07-08."""

    def __init__(self, num_classes, dropout=0.2):
        super().__init__()
        self.cnn = nn.Sequential(
            ResidualBlock(1, 32), nn.MaxPool2d(2, 2),
            ResidualBlock(32, 64), nn.MaxPool2d(2, 2),
            ResidualBlock(64, 128), nn.MaxPool2d((2, 1)),
            nn.Dropout2d(dropout),
            ResidualBlock(128, 128), nn.MaxPool2d((2, 1)),
            nn.Dropout2d(dropout),
            nn.Conv2d(128, 256, (2, 1), 1, 0), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.rnn = nn.LSTM(256, 128, num_layers=2, bidirectional=True, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        feat = self.cnn(x)
        feat = feat.squeeze(2).permute(0, 2, 1)
        seq, _ = self.rnn(feat)
        seq = self.dropout(seq)
        return self.fc(seq).log_softmax(2)


checkpoint = torch.load(MODELS_DIR / 'crnn_residual_aug_elastic.pt', map_location=device)
model = CRNNResidual(NUM_CLASSES, dropout=0.2).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

## Beam search + character n-gram LM (identical to `notebooks/08_decoding_strategies.ipynb`)

Reused as-is from notebook 08 (already validated there: known-answer smoke
test, brute-force CTC forward-score check, and the alpha sweep that found
alpha=0.35). Nothing new here - just re-running the already-confirmed best
decoding config to get fresh predictions to analyze.

In [ ]:
NEG_INF = float('-inf')


def _logsumexp(*args):
    m = max(args)
    if m == NEG_INF:
        return NEG_INF
    return m + math.log(sum(math.exp(a - m) for a in args))


class CharNgramLM:
    """Character n-gram LM, add-k smoothed, trained on this project's own
    fit-split Target text (Amodei et al. 2016 fusion formula - see
    RESOURCES.md). Identical to notebook 08."""

    def __init__(self, texts, n=3, k=1e-3):
        self.n = n
        self.k = k
        self.ngrams = defaultdict(Counter)
        self.context_counts = Counter()
        self.vocab = set()
        for text in texts:
            padded = '^' * (n - 1) + text + '$'
            for i in range(len(padded) - n + 1):
                context = padded[i:i + n - 1]
                char = padded[i + n - 1]
                self.ngrams[context][char] += 1
                self.context_counts[context] += 1
                self.vocab.add(char)
        self.vocab_size = len(self.vocab)

    def get_log_prob(self, char, context):
        if len(context) > self.n - 1:
            context = context[-(self.n - 1):]
        elif len(context) < self.n - 1:
            context = '^' * (self.n - 1 - len(context)) + context
        total = self.context_counts[context]
        count = self.ngrams[context][char]
        prob = (count + self.k) / (total + self.k * self.vocab_size)
        return math.log(prob)


def beam_search_decode(log_probs, input_length, beam_width=10, top_k=25, lm=None, alpha=0.0):
    """CTC prefix beam search with optional LM shallow fusion (Graves et
    al. 2006; Amodei et al. 2016; algorithm per Hannun 2017 - see
    RESOURCES.md). Identical to notebook 08's already-validated version."""
    log_probs = log_probs[:input_length]
    T, C = log_probs.shape
    k = min(top_k, C)
    beams = {(): (0.0, NEG_INF)}

    for t in range(T):
        topk_logp, topk_idx = log_probs[t].topk(k)
        next_beams = {}

        for prefix, (p_b, p_nb) in beams.items():
            p_total = _logsumexp(p_b, p_nb)
            end_char = prefix[-1] if prefix else None
            prefix_str = ''.join(idx_to_char[x] for x in prefix)

            for log_p, c in zip(topk_logp.tolist(), topk_idx.tolist()):
                if c == BLANK:
                    nb = next_beams.get(prefix, (NEG_INF, NEG_INF))
                    next_beams[prefix] = (_logsumexp(nb[0], p_total + log_p), nb[1])
                elif c == end_char:
                    nb = next_beams.get(prefix, (NEG_INF, NEG_INF))
                    next_beams[prefix] = (nb[0], _logsumexp(nb[1], p_nb + log_p))
                    lm_score = alpha * lm.get_log_prob(idx_to_char[c], prefix_str) if lm else 0.0
                    new_prefix = prefix + (c,)
                    nb2 = next_beams.get(new_prefix, (NEG_INF, NEG_INF))
                    next_beams[new_prefix] = (nb2[0], _logsumexp(nb2[1], p_b + log_p + lm_score))
                else:
                    lm_score = alpha * lm.get_log_prob(idx_to_char[c], prefix_str) if lm else 0.0
                    new_prefix = prefix + (c,)
                    nb2 = next_beams.get(new_prefix, (NEG_INF, NEG_INF))
                    next_beams[new_prefix] = (nb2[0], _logsumexp(nb2[1], p_total + log_p + lm_score))

        beams = dict(sorted(next_beams.items(), key=lambda kv: _logsumexp(*kv[1]), reverse=True)[:beam_width])

    best_prefix = max(beams.items(), key=lambda kv: _logsumexp(*kv[1]))[0]
    return ''.join(idx_to_char[c] for c in best_prefix)


def beam_decode_batch(log_probs, input_lengths, beam_width=10, lm=None, alpha=0.0):
    return [beam_search_decode(log_probs[b], input_lengths[b], beam_width=beam_width, lm=lm, alpha=alpha)
            for b in range(log_probs.size(0))]


lm = CharNgramLM(fit_df['Target'].tolist(), n=4, k=1e-3)

## Run the pipeline on the val sample

Sanity check: this should reproduce `score ~= 0.826` (small run-to-run
float noise is expected, e.g. batch-order/padding, not a bug).

In [ ]:
refs, hyps = [], []
with torch.no_grad():
    for imgs, targets_cat, input_lengths, target_lengths, texts in val_dl:
        imgs = imgs.to(device)
        log_probs = model(imgs).cpu()
        hyps.extend(beam_decode_batch(log_probs, input_lengths, beam_width=10, lm=lm, alpha=0.35))
        refs.extend(texts)

score = combined_score(refs, hyps)
print(f"score {score['score']:.4f} (expected ~0.826)")
score

## Build the per-line error table (raw Levenshtein distance + edit-op breakdown)

In [ ]:
ids = val_sample['ID'].tolist()
rows = []
for id_, ref, hyp in zip(ids, refs, hyps):
    ops = Levenshtein.editops(ref, hyp)
    n_sub = sum(1 for op in ops if op.tag == 'replace')
    n_ins = sum(1 for op in ops if op.tag == 'insert')
    n_del = sum(1 for op in ops if op.tag == 'delete')
    rows.append({
        'ID': id_, 'ref': ref, 'hyp': hyp,
        'dist': Levenshtein.distance(ref, hyp),
        'n_sub': n_sub, 'n_ins': n_ins, 'n_del': n_del,
    })

error_df = pd.DataFrame(rows).sort_values('dist', ascending=False).reset_index(drop=True)
error_df['dist'].describe()

In [ ]:
# Confirms the "47% of total error concentrated in dist>10 lines" claim
# from the post-processing investigation, on this fresh run.
catastrophic = error_df[error_df['dist'] > 10].copy()
print(f"{len(catastrophic)}/{len(error_df)} lines with dist > 10, "
      f"{catastrophic['dist'].sum()}/{error_df['dist'].sum()} of total edit distance "
      f"({catastrophic['dist'].sum() / error_df['dist'].sum():.0%})")
catastrophic[['ID', 'dist', 'ref', 'hyp']]

---

# Word-level lexicon opportunity

The line-by-line review above showed something the earlier character-
substitution scan missed: several mid-range-error lines aren't garbled
letter-by-letter, they're garbled at the *word* level - the model outputs
a fluent-looking but non-existent word (`suney`, `tene`, `Portesgne`
instead of `surveyor`, `tenne`, `Fortesque`). A character-confusion
dictionary can't catch that; a word-level lexicon check might.

Before building anything (lexicon-constrained decoding, post-hoc word
correction, etc.) - measure how big this opportunity actually is, same
discipline as the character-level scan: for every word the pipeline
predicted on the 120-row val sample, is it a word that actually appears
somewhere in the *fit* split's vocabulary (no val leakage), and if not,
how close (edit distance) is the nearest real word? If most
out-of-vocabulary predictions sit at distance 1-2 from a real word, a
lexicon-based correction has real headroom. If they're far from anything
real, it won't help (same conclusion the character-level scan reached for
the catastrophic lines).

In [ ]:
from rapidfuzz import process
from rapidfuzz.distance import Levenshtein as _Lev

import re

WORD_RE = re.compile(r"[A-Za-z']+")


def words_of(text):
    """Lowercased alphabetic tokens only - strips punctuation/digits/markup
    chars (e.g. '#', '^', '~') so the lexicon check isn't dominated by
    trivial punctuation mismatches."""
    return [w.lower() for w in WORD_RE.findall(text)]


# Vocabulary built from the FIT split only - val_sample's own words would
# leak the answer into the "is this word real" check.
fit_vocab = set()
for text in fit_df['Target']:
    fit_vocab.update(words_of(text))

len(fit_vocab)

In [ ]:
vocab_list = list(fit_vocab)

hyp_word_rows = []
for id_, hyp in zip(ids, hyps):
    for w in words_of(hyp):
        hyp_word_rows.append({'ID': id_, 'word': w})

hyp_words_df = pd.DataFrame(hyp_word_rows)
hyp_words_df['in_vocab'] = hyp_words_df['word'].isin(fit_vocab)

in_vocab_rate = hyp_words_df['in_vocab'].mean()
print(f"{hyp_words_df['in_vocab'].sum()}/{len(hyp_words_df)} predicted words "
      f"are in the fit-split vocabulary ({in_vocab_rate:.1%})")

In [ ]:
oov_words = sorted(hyp_words_df.loc[~hyp_words_df['in_vocab'], 'word'].unique())
print(f"{len(oov_words)} distinct out-of-vocabulary predicted words")

nearest = {}
for w in oov_words:
    match = process.extractOne(w, vocab_list, scorer=_Lev.distance)
    nearest[w] = match  # (matched_word, distance, index)

oov_df = pd.DataFrame([
    {'word': w, 'nearest_vocab_word': m[0], 'distance': m[1]}
    for w, m in nearest.items()
]).sort_values('distance')

oov_df['distance'].value_counts().sort_index()

In [ ]:
# How much of the OOV set sits close enough to a real word to plausibly
# be a lexicon-correctable misread, vs. far enough that it's likely a
# genuinely different (or catastrophically misread) word?
close = oov_df[oov_df['distance'] <= 2]
far = oov_df[oov_df['distance'] > 2]
print(f"distance <=2 (plausibly correctable): {len(close)}/{len(oov_df)} "
      f"({len(close) / len(oov_df):.1%})")
print(f"distance >2 (likely not a simple misread): {len(far)}/{len(oov_df)} "
      f"({len(far) / len(oov_df):.1%})")
close.head(30)

## Interpretation

87.5% of predicted words are already in the fit-split vocabulary. Of the
159 out-of-vocabulary words, 143 (90%) sit within edit distance 2 of a
real vocabulary word - a real ceiling, bigger than anything the character-
level scan found.

But a manual spot-check of the distance-1 matches showed the naive
nearest-neighbor lookup is unreliable without context:
- Some are clearly right (`precited`->`recited`, `litte`->`little`).
- Some pick the wrong candidate when two real words tie at distance 1
  (`pround`->`pound`, when the actual reference word `ground` is equally
  distance 1).
- Proper nouns are a blind spot (`haved`->`saved`, when the reference is
  the name `David`, itself distance 2 - a worse match, but the rule can't
  know that).
- Genuine period spelling can get "corrected" toward a modern word that's
  actually further from the (also period-spelled) reference
  (`peaceablye`->`peaceably`), re-introducing exactly the kind of
  normalization this project already decided against in notebook 04.

Decision: don't hand-pick which matches "look right" (that's tuning to
this sample by eye, won't generalize) - define an objective, reproducible
rule instead and measure its net effect. See the gated test below.

---

# Apply an objective "confident correction" rule and measure the effect

The qualitative spot-check above was mixed: some distance-1 matches were
clearly right (`precited`->`recited`), others clearly wrong
(`pround`->`pound`, should be `ground`; `haved`->`saved`, should be the
proper noun `David`), and one showed a specific new risk - `peaceablye`
(a valid period spelling, distance 1 from the *reference*) would get
"corrected" to the modern `peaceably`, moving it further from a reference
that may itself use non-modern spelling.

Hand-picking which of the distance-1 matches "look right" from the sample
above isn't a real rule - it's tuning to this specific val sample by eye,
which won't generalize. Instead, define an objective, reproducible filter
and let the actual score decide:

1. **`len(word) >= 5`** - short words (`la`, `int`) were the least
   reliable matches in the sample; an edit distance of 1 is much weaker
   evidence on a 2-3 letter word than on an 8-letter one.
2. **Unique nearest neighbor only** - skip the correction if more than one
   vocabulary word ties for the minimum distance (this alone would have
   caught `pround`, where `pound` and `ground` are both distance 1 - an
   unresolvable tie without more context than a lexicon lookup has).

The `peaceablye`/period-spelling risk isn't filterable by either rule and
is left as residual noise - if it's rare, the net effect should still be
measurable and net positive or negative either way.

In [ ]:
MIN_WORD_LEN = 5

def nearest_matches(word, vocab):
    dists = [(v, _Lev.distance(word, v)) for v in vocab]
    min_d = min(d for _, d in dists)
    best = [v for v, d in dists if d == min_d]
    return min_d, best


confident_corrections = {}
for w in oov_words:
    if len(w) < MIN_WORD_LEN:
        continue
    min_d, best = nearest_matches(w, vocab_list)
    if min_d == 1 and len(best) == 1:
        confident_corrections[w] = best[0]

print(f"{len(confident_corrections)} confident corrections out of "
      f"{len(oov_words)} OOV words ({len(oov_words) - len(confident_corrections)} "
      f"skipped: too short or ambiguous nearest neighbor)")
confident_corrections

In [ ]:
def apply_corrections(hyp):
    """Applies confident_corrections, but skips a match if it's
    capitalized AND not the first word of the line - mid-sentence
    capitalization in this corpus is almost always a proper noun
    (Bearnes, Shamon, Griffin, Anderson), and this project's own corpus
    already shows inconsistent proper-noun spelling (Griffen/Griffin/
    Griffine all appear), so 'correcting' toward the nearest known
    spelling is unprincipled for names specifically. Sentence-initial
    capitals (e.g. 'Andian' -> 'Indian') are still corrected normally."""
    result = []
    last_end = 0
    seen_first_word = False
    for m in WORD_RE.finditer(hyp):
        start, end = m.span()
        result.append(hyp[last_end:start])
        w = m.group()
        lw = w.lower()
        is_first_word = not seen_first_word
        seen_first_word = True

        if lw in confident_corrections and not (w[0].isupper() and not is_first_word):
            corrected = confident_corrections[lw]
            result.append(corrected.capitalize() if w[0].isupper() else corrected)
        else:
            result.append(w)
        last_end = end
    result.append(hyp[last_end:])
    return ''.join(result)


corrected_hyps = [apply_corrections(h) for h in hyps]

before = combined_score(refs, hyps)
after = combined_score(refs, corrected_hyps)
print('before (beam+LM, no correction):', before)
print('after  (+ lexicon correction, proper-noun-safe):', after)
print(f"delta score: {after['score'] - before['score']:+.4f}")

In [ ]:
# Line-by-line: which val lines actually changed, and did each one move
# closer to or further from its reference?
changed = []
for id_, ref, h, c in zip(ids, refs, hyps, corrected_hyps):
    if h != c:
        d_before = _Lev.distance(ref, h)
        d_after = _Lev.distance(ref, c)
        changed.append({
            'ID': id_, 'ref': ref, 'before': h, 'after': c,
            'dist_before': d_before, 'dist_after': d_after,
            'delta': d_after - d_before,
        })

changed_df = pd.DataFrame(changed).sort_values('delta')
print(f"{len(changed_df)} lines modified "
      f"({(changed_df['delta'] < 0).sum()} improved, "
      f"{(changed_df['delta'] > 0).sum()} regressed, "
      f"{(changed_df['delta'] == 0).sum()} unchanged distance)")
changed_df

## Gate check - result: NOT ADOPTED (negative/inconclusive)

Two versions tested against the 0.826 reference:

- Unfiltered correction rule: `score = 0.8281` (+0.0015), but corrupted 2
  previously 100%-correct lines by "fixing" rare-but-correct proper nouns
  (`Bearnes`->`Barnes`, `Shamon`->`Shymon`).
- Proper-noun-safe refinement (skip mid-sentence capitalized words):
  `score = 0.8277` (+0.0011) - no longer breaks those two lines, but the
  net gain shrinks to the same order of magnitude as the beam-search-alone
  control from notebook 08 (+0.001), already established in this project
  as noise, not a real improvement.

Root cause: the fit split's vocabulary (~4,484 words from 3,280 lines) is
too sparse to reliably distinguish a genuine OCR error from a valid but
rare period spelling or proper noun not covered by that sample (e.g. the
model correctly predicted the period spelling `Containeing`, which
doesn't appear anywhere in the training text - only `Conteyneing` does -
so the rule "corrected" a right answer into the wrong one). Would need a
substantially larger or genuinely period-appropriate dictionary to have
real headroom; not available here.

**Reference pipeline stays at 0.826** (`crnn_residual_aug_elastic.pt` +
beam search + 4-gram char LM, no post-processing). Logged in
`README.md`'s Progress/Next steps.